In [1]:
import numpy as np
import pandas as pd

In [4]:
# Expost Attribution

returns_input = pd.read_csv("/Users/fuyuxuan/Downloads/test11_1_returns.csv")
weights_input= pd.read_csv("/Users/fuyuxuan/Downloads/test11_1_weights.csv")

# Convert to numpy array
R = returns_input.values
w0 = weights_input.iloc[:, 0].values

# Number of periods and assets
T, n= R.shape

# Total Return
asset_total_return = np.prod(1 + R, axis=0) - 1.0

# Portfolio buy-and-hold total return
portfolio_total_return = np.sum(w0 * asset_total_return)

# Return Attribution
# Compute the beginning-of-period weights under buy-and-hold
w_t = w0.copy()
aseet_contribution_to_portfolio = []
portfolio_return = []

for t in range(T):
    each_asset_contribution = w_t * R[t, :]
    aseet_contribution_to_portfolio.append(each_asset_contribution)
    all_asset_contribution = np.sum(each_asset_contribution)
    portfolio_return.append(all_asset_contribution)

    # Update weights to next period
    w_t = w_t * (1 + R[t, :]) / (1 + all_asset_contribution)

aseet_contribution_to_portfolio = np.array(aseet_contribution_to_portfolio)
portfolio_return = np.array(portfolio_return)

Rp = np.prod(1 + portfolio_return) - 1.0                             # Total portfolio return in the full horizon
k = np.log(1 + Rp) / Rp if abs(Rp) > 1e-12 else 1.0    # compute Carino linking coefficient for full-period portfolio return
k_t = np.where(np.abs(portfolio_return) > 1e-12, np.log(1 + portfolio_return) / portfolio_return, 1.0) # compute Carino coefficient for each period

return_attr = np.sum(aseet_contribution_to_portfolio * (k_t / k).reshape(-1, 1), axis=0)

# Vol Attribution
# sample covariance contribution using time-varying contributions
vol_port = np.std(portfolio_return, ddof=1)
vol_attr = np.array([
    np.cov(aseet_contribution_to_portfolio[:, i], portfolio_return, ddof=1)[0, 1] / vol_port
    for i in range(n)
])

output = pd.DataFrame({
    "Value": ["TotalReturn", "Return Attribution", "Vol Attribution"],
    "x1": [asset_total_return[0], return_attr[0], vol_attr[0]],
    "x2": [asset_total_return[1], return_attr[1], vol_attr[1]],
    "x3": [asset_total_return[2], return_attr[2], vol_attr[2]],
    "Portfolio": [portfolio_total_return, np.sum(return_attr), vol_port]
})

print(output)

                Value        x1        x2        x3  Portfolio
0         TotalReturn -0.221446 -0.016008  0.301467   0.081098
1  Return Attribution -0.065513 -0.002220  0.148831   0.081098
2     Vol Attribution -0.000620  0.002827  0.012580   0.014787
